In [4]:
conda create -n ros_env python=3.11 jupyter ipykernel ipywidgets ros-humble-desktop -c robostack-humble -c conda-forge --strict-channel-priority -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - robostack-humble
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3/envs/ros_env

  added / updated specs:
    - ipykernel
    - ipywidgets
    - jupyter
    - python=3.11
    - ros-humble-desktop


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |       7_kmp_llvm           8 KB  conda-forge
    adwaita-icon-theme-49.0    |           unix_0         617 KB  conda-forge
    aiohappyeyeballs-2.7.1     |     pyhd8ed1ab_0          24 KB  conda-forge
    aiohttp-3.14.2             |  py311h6771f6e_0         1.

In [2]:
# Starting code for turtlebot
import rclpy
if not rclpy.ok():
    rclpy.init()

# creating a function
from robot_wrapper import Robot
# Instantiate Wrapper. The subscribers and publishes the relevant topics and connects the actions.
robot = Robot()

In [ ]:
def robot_square():
    robot.set_cmd_vel(1,0, 0.5)
    robot.set_cmd_vel(0, 1, 1.5) # 90 degrees left so -1 rotational velocity would be right
    robot.set_cmd_vel(1,0, 0.5)
    robot.set_cmd_vel(0, 1, 1.5)
    robot.set_cmd_vel(1,0, 0.5)
    robot.set_cmd_vel(0, 1, 1.5)
    robot.set_cmd_vel(1,0, 0.5)

In [ ]:
robot_square() #Here I am playing around with how the robot moves

In [ ]:
'''
Code that I want to write in written words:
distance_to_obstacle = get distance from lidar
if distance_to_obstacle <= 10 cm:
    robot.set_cmd_vel(0,0,0)
    break
'''

In [ ]:
# Attempt 1

In [ ]:
import rclpy.node
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist

In [ ]:
class LidarObserver(Node):
    def __init__(self):
        super().__init__('lidar_observer')
        # Subscribing to the LiDAR
        self.subscription = self.create_subscription(
            LaserScan,
            '/scan',
            self.avoiding_obstacles,
            10
        )
        # Create a publisher to send velocity commands to the robot
        self.publisher = self.create_publisher(Twist, '/cmd_vel', 10)

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        
        if not ranges:
            return

        # Get distance directly in front (0 degrees)
        front_distance = ranges[0]
        
        # Get distance to the left (90 degrees)
        left_distance = ranges[90]
        
        # Get distance to the right (270 degrees)
        right_distance = ranges[270]
        
        # Get distance to the back (180 degrees)
        back_distance = ranges[180]

        # Creating A Twist message (default linear and angular elements will be 0.0)
        twist = Twist()

        if 0.0 < front_distance <= 0.5:
            self.get_logger().info(f"Obstacle detected ahead ({front_distance:.2f}m)! STOPPING!")
            
            # Explicitly setting linear and angular velocities to 0
            twist.linear.x = 0.0
            twist.angular.z = 0.0
            
            # Publish the stop command to the robot
            self.publisher.publish(twist)
            print("There is an obstacle! I must stop!")
            return
        else:
            twist.linear.x = 0.15 
            twist.angular.z = 0.0
            self.publisher.publish(twist)
            # Path is clear; do nothing or add driving logic here
            pass

In [ ]:
def main(args=None):
    # Check if a ROS 2 context is already running in this notebook session (which it most likely will be)
    if not rclpy.ok():
        rclpy.init(args=args)
    else:
        # If it's already running, shutdown the old one and start fresh
        rclpy.shutdown()
        rclpy.init(args=args)

    node = LidarObserver()
    
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        print("Shutting down node safely...")
    finally:
        # Cleaning up the node and context so the next run works
        node.destroy_node()
        rclpy.shutdown()

In [ ]:
if __name__ == '__main__':
    main()